Se monta Google Drive para acceder al dataset desde Colab.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Se importan las librerías necesarias para manipulación de datos, preprocesamiento, gráficos, optimización y evaluación del modelo. %matplotlib inline permite mostrar los gráficos directamente en el cuaderno.

In [2]:
# utilizado para la manipulación de directorios y rutas
import os
import pandas as pd
# Cálculo científico y vectorial para python
import numpy as np

# Libreria para graficos
from matplotlib import pyplot

from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler
from scipy import optimize

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
sns.set()


# Modulo de optimizacion en scipy
from scipy import optimize

# modulo para cargar archivos en formato MATLAB
# from scipy.io import loadmat

# le dice a matplotlib que incruste gráficos en el cuaderno
%matplotlib inline

Se carga el dataset desde Google Drive, se revisan las columnas y se limpia la columna Make. Luego se codifica Make en números para usarla como variable objetivo y. Las columnas seleccionadas se usan como entradas X, y se define el número de clases únicas num_labels.

In [3]:
# Cargar dataset
data = pd.read_csv('/content/drive/MyDrive/IA/DATASETS/Electric_Vehicle_Population_Data (1).csv')

# Revisar nombres de columnas
print(data.columns)

# Limpiar columna Make y convertir a número
data['Make'] = data['Make'].astype(str).str.strip().str.upper()
data['Make_num'], marcas_unicas = pd.factorize(data['Make'], sort=True)

# Definir X y y
# X: todas las columnas que quieres como input (por ejemplo numéricas)
X = data[['Model Year', 'Electric Range', 'Base MSRP', 'Legislative District', '2020 Census Tract']].values

# y: la columna codificada
y = data['Make_num'].values

# Establecer el número total de etiquetas (marcas únicas)
num_labels = len(marcas_unicas)


# print(data['y'])
# print(data['y'].ravel())
# print(data['X'])
#X = data[:, 1:]
#y = data[:, 0]
# X, y = data['X'], data['y'].ravel()
# print(X)
# print(y)
# Si quieres establecer algún valor de target como 0
# establecer el dígito cero en 0, en lugar del 10 asignado a este conjunto de datos
# Esto se hace debido a que el conjunto de datos se utilizó en MATLAB donde no hay índice 0
# y[y == 3] = 0 # Comentado para no modificar y y mantener todas las etiquetas originales
# print(y)

m = y.size

Index(['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year',
       'Make', 'Model', 'Electric Vehicle Type',
       'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Base MSRP', 'Legislative District', 'DOL Vehicle ID',
       'Vehicle Location', 'Electric Utility', '2020 Census Tract'],
      dtype='object')


In [4]:
m, n = data.shape
print(f"Número de ejemplos (m): {m}")
print(f"Número de columnas (n): {n}")

Número de ejemplos (m): 257635
Número de columnas (n): 18


Se calcula el número de clases únicas en la variable objetivo (Make_num) para confirmar la cantidad de categorías que tendrá el modelo de regresión logística multiclase.

In [5]:
# Usando la columna numérica
num_marcas = data['Make_num'].nunique()
print(f"Número de marcas únicas: {num_marcas}")

Número de marcas únicas: 46


Se crea un diccionario que asigna un número a cada marca, facilitando la codificación de las clases para la regresión logística multiclase.

In [6]:
# Crear un diccionario marca → número
marca_dict = dict(zip(marcas_unicas, range(len(marcas_unicas))))

# Mostrar todas las marcas con su número
print("Marca → Make_num asignado:")
for marca, num in marca_dict.items():
    print(f"{marca} → {num}")

Marca → Make_num asignado:
ACURA → 0
ALFA ROMEO → 1
AUDI → 2
AZURE DYNAMICS → 3
BENTLEY → 4
BMW → 5
BRIGHTDROP → 6
CADILLAC → 7
CHEVROLET → 8
CHRYSLER → 9
DODGE → 10
FIAT → 11
FISKER → 12
FORD → 13
GENESIS → 14
GMC → 15
HONDA → 16
HYUNDAI → 17
JAGUAR → 18
JEEP → 19
KIA → 20
LAMBORGHINI → 21
LAND ROVER → 22
LEXUS → 23
LINCOLN → 24
LUCID → 25
MAZDA → 26
MERCEDES-BENZ → 27
MINI → 28
MITSUBISHI → 29
MULLEN AUTOMOTIVE INC. → 30
NISSAN → 31
POLESTAR → 32
PORSCHE → 33
RAM → 34
RIVIAN → 35
ROLLS-ROYCE → 36
SMART → 37
SUBARU → 38
TESLA → 39
TH!NK → 40
TOYOTA → 41
VINFAST → 42
VOLKSWAGEN → 43
VOLVO → 44
WHEEGO ELECTRIC CARS → 45


In [7]:
print(X[0,:])
print(y)
print(y[:50])   # mostrar los primeros 50
print(y[-50:])  # mostrar los últimos 50

[2.01900000e+03 2.20000000e+02 0.00000000e+00 1.50000000e+01
 5.30770015e+10]
[39 19 20 ...  5 41 17]
[39 19 20  5 31 31 39 39 31 33 39  5  8 39 32 20 31 31 31 16 20 39 39 39
 39 39 39 39 39  5 31 39 39 44 31 39  5 39 39  8 44  2 31 31 13  2  2 39
 39  8]
[ 5 44 13 20 39 20 39  8  2 39 31 13 41 20 39 39  5 39 39 39 13 31 39 19
 39 31 39 39 17 13 39  8 13 39 39 39 39  8 44 39  5 39 39 33 39  9 39  5
 41 17]


Se identifican columnas numéricas y categóricas. Las categóricas se convierten en variables numéricas usando One-Hot Encoding, permitiendo que el modelo las use como entradas.

In [8]:
# Columnas categóricas a convertir a números
# Columnas numéricas
numerical_cols = ['Model Year', 'Electric Range', 'Base MSRP', 'Legislative District', '2020 Census Tract']

# Columnas categóricas
categorical_cols = ['State', 'Model', 'Electric Vehicle Type',
                    'Clean Alternative Fuel Vehicle (CAFV) Eligibility',
                    'Electric Utility']

# One-hot encoding actualizado para versiones recientes de sklearn
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = encoder.fit_transform(data[categorical_cols])

print("Shape de X_cat:", X_cat.shape)

Shape de X_cat: (257635, 311)


Se define una función para normalizar las columnas numéricas, restando la media y dividiendo por la desviación estándar, lo que ayuda a que el modelo aprenda mejor.

In [9]:
# Normalizar columnas numéricas
def  featureNormalize(X):
    X_norm = X.copy()
    mu = np.zeros(X.shape[1])
    sigma = np.zeros(X.shape[1])

    mu = np.mean(X, axis = 0)
    sigma = np.std(X, axis = 0)
    X_norm = (X - mu) / sigma

    return X_norm, mu, sigma

Se codifican las columnas categóricas como números, se normalizan las columnas numéricas y luego se combinan ambas en X_norm, que será la entrada final del modelo. Se revisa la forma y algunos registros para confirmar que todo esté correcto.

In [10]:
# Convertir columnas categóricas a números
for col in categorical_cols:
    data[col + '_num'], _ = pd.factorize(data[col], sort=True)

# Normalizar columnas numéricas
X_num = data[numerical_cols].fillna(data[numerical_cols].mean()).values
X_num_norm, mu, sigma = featureNormalize(X_num)

# Ahora sí existen las columnas "_num"
categorical_cols_num = [col + '_num' for col in categorical_cols]
X_cat = data[categorical_cols_num].values

# Combinar numéricas normalizadas + categóricas codificadas
X_norm = np.hstack([X_num_norm, X_cat])

# Revisar
print("Shape X_norm:", X_norm.shape)
print("Primer registro X_norm:", X_norm[0,:])
print("Primeros 50 y:", y[:50])
print("Ultimos 50 y:", y[-50:])

Shape X_norm: (257635, 10)
Primer registro X_norm: [-8.97254627e-01  2.16669755e+00 -1.00791667e-01 -9.34897949e-01
  6.36752729e-02  4.70000000e+01  1.00000000e+02  0.00000000e+00
  0.00000000e+00  6.50000000e+01]
Primeros 50 y: [39 19 20  5 31 31 39 39 31 33 39  5  8 39 32 20 31 31 31 16 20 39 39 39
 39 39 39 39 39  5 31 39 39 44 31 39  5 39 39  8 44  2 31 31 13  2  2 39
 39  8]
Ultimos 50 y: [ 5 44 13 20 39 20 39  8  2 39 31 13 41 20 39 39  5 39 39 39 13 31 39 19
 39 31 39 39 17 13 39  8 13 39 39 39 39  8 44 39  5 39 39 33 39  9 39  5
 41 17]


Se codifican las columnas categóricas usando One-Hot Encoding (get_dummies) y también con números enteros (factorize) para tener dos opciones de representación de las categorías según lo que necesite el modelo.

In [11]:
# Factorizar columnas categóricas
X_cat = pd.get_dummies(data[['State','Model','Electric Vehicle Type',
                             'Clean Alternative Fuel Vehicle (CAFV) Eligibility',
                             'Electric Utility']], drop_first=True)
data['State_num'], _ = pd.factorize(data['State'], sort=True)
data['Model_num'], _ = pd.factorize(data['Model'], sort=True)
data['Electric_Vehicle_Type_num'], _ = pd.factorize(data['Electric Vehicle Type'], sort=True)
data['Clean_Alternative_Fuel_Vehicle__CAFV__Eligibility_num'], _ = pd.factorize(data['Clean Alternative Fuel Vehicle (CAFV) Eligibility'], sort=True)
data['Electric_Utility_num'], _ = pd.factorize(data['Electric Utility'], sort=True)

Se normalizan las columnas numéricas y se combinan con las categóricas codificadas para crear la matriz de entradas X_norm.

In [12]:
# llama featureNormalize con los datos cargados
categorical_cols = ['State', 'Model', 'Electric Vehicle Type',
                    'Clean Alternative Fuel Vehicle (CAFV) Eligibility',
                    'Electric Utility']

numerical_cols = ['Model Year', 'Electric Range', 'Base MSRP', 'Legislative District', '2020 Census Tract']
X_num = data[numerical_cols].fillna(data[numerical_cols].mean()).values
X_num_norm, mu, sigma = featureNormalize(X_num)


categorical_cols_num = ['State_num', 'Model_num', 'Electric_Vehicle_Type_num',
                        'Clean_Alternative_Fuel_Vehicle__CAFV__Eligibility_num',
                        'Electric_Utility_num']
X_cat = data[categorical_cols_num].values  # ahora sí son números

# Combinar numéricas normalizadas + categóricas
X_norm = np.hstack([X_num_norm, X_cat])



Se aplica One-Hot Encoding a las columnas categóricas para convertirlas en variables binarias que el modelo pueda usar.

In [13]:
#One-Hot Encoding para categóricas
encoder = OneHotEncoder(handle_unknown='ignore')
X_cat = encoder.fit_transform(data[categorical_cols]).toarray()

In [14]:
print("Shape X_norm:", X_norm.shape)
print("Primer registro de X_norm:", X_norm[0,:])
print("Primeros 50 valores de y:", y[:50])

Shape X_norm: (257635, 10)
Primer registro de X_norm: [-8.97254627e-01  2.16669755e+00 -1.00791667e-01 -9.34897949e-01
  6.36752729e-02  4.70000000e+01  1.00000000e+02  0.00000000e+00
  0.00000000e+00  6.50000000e+01]
Primeros 50 valores de y: [39 19 20  5 31 31 39 39 31 33 39  5  8 39 32 20 31 31 31 16 20 39 39 39
 39 39 39 39 39  5 31 39 39 44 31 39  5 39 39  8 44  2 31 31 13  2  2 39
 39  8]


In [15]:
print(X_norm[0,:])
print(y[:50])

[-8.97254627e-01  2.16669755e+00 -1.00791667e-01 -9.34897949e-01
  6.36752729e-02  4.70000000e+01  1.00000000e+02  0.00000000e+00
  0.00000000e+00  6.50000000e+01]
[39 19 20  5 31 31 39 39 31 33 39  5  8 39 32 20 31 31 31 16 20 39 39 39
 39 39 39 39 39  5 31 39 39 44 31 39  5 39 39  8 44  2 31 31 13  2  2 39
 39  8]


Se agrega una columna de unos a X_norm para el término de intercepción del modelo de regresión logística.

In [16]:
# Configurar la matriz adecuadamente, y agregar una columna de unos que corresponde al termino de intercepción.
m, n = X_norm.shape # X_norm ya tiene la forma correcta
# Agraga el termino de intercepción a A
X = np.concatenate([np.ones((m, 1)), X_norm], axis=1) # Esto se hará en OneVsAllOM si es necesario

#X = X_norm # Usamos X_norm directamente como input para OneVsAllOM


In [17]:
#División entrenamiento/prueba 80/20
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (206108, 10) Test shape: (51527, 10)


**Vectorización de regresión logística**

Se define la función sigmoide, que transforma cualquier valor a un rango entre 0 y 1, usada en regresión logística.

In [18]:
def sigmoid(z):
    """
    Calcula la sigmoide de z.
    """
    return 1.0 / (1.0 + np.exp(-z))

Se define la función de costo de regresión logística, que mide qué tan bien el modelo predice los datos.

In [19]:
def calcularCosto(theta, X, y):
    # Inicializar algunos valores utiles
    m = y.size  # numero de ejemplos de entrenamiento

    J = 0
    h = sigmoid(X.dot(theta))
    J = (1 / m) * np.sum(-y.dot(np.log(h)) - (1 - y).dot(np.log(1 - h)))

    return J

Se implementa el descenso por gradiente para actualizar theta y minimizar el costo iterativamente.

In [20]:
def descensoGradiente(theta, X, y, alpha, num_iters):
    # Inicializa algunos valores
    m = y.shape[0] # numero de ejemplos de entrenamiento

    # realiza una copia de theta, el cual será acutalizada por el descenso por el gradiente
    theta = theta.copy()
    J_history = []

    for i in range(num_iters):
        h = sigmoid(X.dot(theta.T))
        theta = theta - (alpha / m) * (h - y).dot(X)

        J_history.append(calcularCosto(theta, X, y))
    return theta, J_history

Se define la función de costo regularizada de regresión logística, que penaliza grandes valores de theta para evitar sobreajuste, y calcula el gradiente para actualizar los parámetros.

In [21]:
def lrCostFunction(theta, X, y, lambda_):
    """
    Calcula el costo de usar theta como parámetro para la regresión logística regularizada y
    el gradiente del costo w.r.t. a los parámetros.

    Parametros
    ----------
    theta : array_like
        Parametro theta de la regresion logistica. Vector de la forma(shape) (n, ). n es el numero de caracteristicas
        incluida la intercepcion

    X : array_like
        Dataset con la forma(shape) (m x n). m es el numero de ejemplos, y n es el numero de
        caracteristicas (incluida la intercepcion).

    y : array_like
        El conjunto de etiquetas. Un vector con la forma (shape) (m, ). m es el numero de ejemplos

    lambda_ : float
        Parametro de regularización.

    Devuelve
    -------
    J : float
        El valor calculado para la funcion de costo regularizada.

    grad : array_like
        Un vector de la forma (shape) (n, ) que es el gradiente de la
        función de costo con respecto a theta, en los valores actuales de theta..
    """
   #  alpha = 0.003
  #   theta = theta.copy()
    # Inicializa algunos valores utiles
    m = y.size
    h = sigmoid(X.dot(theta))
    # convierte las etiquetas a valores enteros si son boleanos
    if y.dtype == bool:
        y = y.astype(int)


    epsilon = 1e-10  # Evita log(0)

    J = 0
    grad = np.zeros(theta.shape)

    h = sigmoid(X.dot(theta.T))

    temp = theta
    temp[0] = 0 # No regularizar el bias

#     J = (1 / m) * np.sum(-y.dot(np.log(h)) - (1 - y).dot(np.log(1 - h)))
    J = (1 / m) * np.sum(-y.dot(np.log(h + epsilon)) - (1 - y).dot(np.log(1 - h + epsilon))) \
        + (lambda_ / (2 * m)) * np.sum(np.square(temp))
    grad = (1 / m) * (h - y).dot(X) + (lambda_ / m) * temp

    return J, grad

**Vectorización regularizada de la regresión logística**

**Clasificacion One-vs-all**

Se entrena un modelo One-vs-All para clasificación multiclase, ajustando un theta por cada clase y mostrando la convergencia del costo durante el entrenamiento.

In [22]:
#entrenamiento
def OneVsAll(X, y, num_labels, lambda_):
    alpha = 0.01
    num_iters = 20000

    m, n = X.shape
    all_theta = np.zeros((num_labels, n + 1))

    # Agrega unos a la matriz X
    Xb = np.concatenate([np.ones((m, 1)), X], axis=1)

    for c in np.arange(num_labels):
        initial_theta = np.zeros(n + 1)
        y_actual = np.where(y == c, 1, 0)
        theta, J_history = descensoGradiente(initial_theta, X, y_actual, alpha, num_iters)
        all_theta[c] = theta
        # Grafica la convergencia del costo
        pyplot.plot(np.arange(len(J_history)), J_history, lw=2)
        pyplot.xlabel('Número de iteraciones')
        pyplot.ylabel('Costo J')


    return all_theta, all_J_history  #ahora está correctamente indentado dentro de la función


OneVsAllOM entrena un modelo One-vs-All usando optimización avanzada (optimize.minimize) para obtener theta de cada clase y mostrar el costo final.

In [ ]:
def OneVsAllOM(X, y, num_labels, lambda_):
    """
    Trains num_labels logistic regression classifiers and returns
    each of these classifiers in a matrix all_theta, where the i-th
    row of all_theta corresponds to the classifier for label i.

    Parameters
    ----------
    X : array_like
        The input dataset of shape (m x n). m is the number of
        data points, and n is the number of features. Note that we
        do not assume that the intercept term (or bias) is in X, however
        we provide the code below to add the bias term to X.

    y : array_like
        The data labels. A vector of shape (m, ).

    num_labels : int
        Number of possible labels.

    lambda_ : float
        The logistic regularization parameter.

    Returns
    -------
    all_theta : array_like
        The trained parameters for logistic regression for each class.
        This is a matrix of shape (K x n+1) where K is number of classes
        (ie. `numlabels`) and n is number of features without the bias.
    """
    # algunas variables utiles
    m, n = X.shape
    all_theta = np.zeros((num_labels, n + 1))

    # Agregar columna de unos para el bias
    Xb = np.concatenate([np.ones((m, 1)), X], axis=1)

    for c in range(num_labels):
        initial_theta = np.zeros(n + 1)

        # Función lambda que devuelve costo y gradiente
        res = optimize.minimize(
            fun=lambda t: lrCostFunction(t, Xb, (y == c).astype(int), lambda_),
            x0=initial_theta,
            jac=True,
            method='CG',
            options={'maxiter': 50}
        )
        # Guardar theta entrenada en la fila correspondiente
        all_theta[c] = res.x

        # Mostrar costo final
        J_final = lrCostFunction(res.x, Xb, (y == c).astype(int), lambda_)[0]
        print(f"Clase {c} entrenada, costo final = {J_final:.4f}")

    return all_theta

Se entrena el modelo One-vs-All con los datos de entrenamiento y regularización lambda_, guardando los parámetros theta de cada clase en all_theta.

In [ ]:
#entrenamiento
# X_train NO tiene columna de unos
lambda_ = 0.1
num_labels = len(np.unique(y))
all_theta = OneVsAllOM(X_train, y_train, num_labels, lambda_)

In [ ]:
print(all_theta)

**Prediccion One-vs-all**

Se define la función para predecir la clase de cada ejemplo usando los parámetros all_theta entrenados en One-vs-All.

In [ ]:
def predictOneVsAll(all_theta, X):
    """
    Devuelve un vector de predicciones para cada ejemplo en la matriz X.
    Tenga en cuenta que X contiene los ejemplos en filas.
    all_theta es una matriz donde la i-ésima fila es un vector theta de regresión logística entrenada para la i-ésima clase.
    Debe establecer p en un vector de valores de 0..K-1 (por ejemplo, p = [0, 2, 0, 1]
    predice clases 0, 2, 0, 1 para 4 ejemplos).

    Parametros
    ----------
    all_theta : array_like
        The trained parameters for logistic regression for each class.
        This is a matrix of shape (K x n+1) where K is number of classes
        and n is number of features without the bias.

    X : array_like
        Data points to predict their labels. This is a matrix of shape
        (m x n) where m is number of data points to predict, and n is number
        of features without the bias term. Note we add the bias term for X in
        this function.

    Devuelve
    -------
    p : array_like
        The predictions for each data point in X. This is a vector of shape (m, ).
    """

    m = X.shape[0];
    #num_labels = all_theta.shape[0]

    # Agrega columna de unos (bias)
    Xb = np.concatenate([np.ones((m, 1)), X], axis=1)
    # Calcula predicción
    p = np.argmax(sigmoid(Xb.dot(all_theta.T)), axis=1)
    return p

    y_pred = predictOneVsAll(all_theta, X)
    train_acc = np.mean(y_pred == y) * 100
    print(f"Precisión conjunto entrenamiento: {train_acc:.2f}%")  # Debe ser ~95.1%

Se entrena el modelo One-vs-All con los datos de entrenamiento y regularización lambda_, obteniendo los parámetros theta para cada clase.

In [ ]:
#entrenamiento
lambda_ = 0.1
all_theta = OneVsAllOM(X_train, y_train, num_labels, lambda_)

In [ ]:
print("X_train.shape:", X_train.shape)
print("X_test.shape:", X_test.shape)
print("all_theta.shape:", all_theta.shape)

In [ ]:
#Predicción
pred_train = predictOneVsAll(all_theta, X_train)
pred_test = predictOneVsAll(all_theta, X_test)
# Calcular precisión en entrenamiento
train_acc = np.mean(pred_train == y_train) * 100
test_acc = np.mean(pred_test == y_test) * 100

print(f"Precisión conjunto entrenamiento: {train_acc:.2f}%")
print(f"Precisión conjunto prueba: {test_acc:.2f}%")

Se grafica el costo final de cada clase entrenada para visualizar la efectividad del modelo en cada categoría.

In [ ]:
#Graficar costo final por clase
costs = [lrCostFunction(all_theta[c], np.concatenate([np.ones((X_train.shape[0],1)), X_train], axis=1),
                        (y_train==c).astype(int), lambda_)[0] for c in range(num_labels)]

pyplot.figure(figsize=(12,5))
pyplot.bar(range(num_labels), costs)
pyplot.xlabel("Clase (Make_num)")
pyplot.ylabel("Costo final")
pyplot.title("Costo final de cada clase One-vs-All")
pyplot.show()

Se verifica que el dataset esté balanceado mostrando cuántos ejemplos tiene cada clase.

In [ ]:
#Revisar balance de clases
class_counts = pd.Series(y).value_counts()
print("Número de ejemplos por clase:\n", class_counts)

**Analizar el desbalance de clases**


Se analiza la distribución de clases usando class_counts para identificar posibles desbalances en el dataset.

In [ ]:
print("Número de ejemplos por clase:\n", class_counts)

**Seleccionar una técnica para manejar el desbalance**

Se aplica SMOTE para balancear las clases, generando ejemplos sintéticos de las clases minoritarias y verificando el nuevo conteo de cada clase.



In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=1) # Set k_neighbors to 1
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Original train shape:", X_train.shape)
print("Resampled train shape:", X_train_resampled.shape)

class_counts_resampled = pd.Series(y_train_resampled).value_counts().sort_index()
print("\nNúmero de ejemplos por clase después de SMOTE:\n", class_counts_resampled)

Se ajusta SMOTE usando k_neighbors=1 para evitar errores con clases muy pequeñas y se balancean las clases generando ejemplos sintéticos.



In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=1) # Set k_neighbors to 1
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Original train shape:", X_train.shape)
print("Resampled train shape:", X_train_resampled.shape)

class_counts_resampled = pd.Series(y_train_resampled).value_counts().sort_index()
print("\nNúmero de ejemplos por clase después de SMOTE:\n", class_counts_resampled)

**Re-entrenar el modelo**

Se re-entrena el modelo One-vs-All usando los datos balanceados por SMOTE para mejorar la predicción en las clases minoritarias.

In [ ]:
# 1. Define the regularization parameter lambda_
lambda_ = 0.1  # You can adjust this value

# 2. Call the OneVsAllOM function using the resampled training data
# 3. Store the trained parameters in the variable all_theta_resampled
all_theta_resampled = OneVsAllOM(X_train_resampled, y_train_resampled, num_labels, lambda_)

**Evaluar el modelo re-entrenado**

Se evalúa la precisión del modelo re-entrenado con los datos balanceados y se compara con la precisión del modelo original para observar mejoras.

In [ ]:
# Utiliza la función predictOneVsAll con los parámetros entrenados con los datos re-balanceados y X_test
pred_test_resampled = predictOneVsAll(all_theta_resampled, X_test)

# Calcula la precisión general del modelo en el conjunto de prueba
test_acc_resampled = np.mean(pred_test_resampled == y_test) * 100

# Imprime la precisión calculada en el conjunto de prueba con el modelo balanceado
print(f"Precisión conjunto prueba (modelo balanceado): {test_acc_resampled:.2f}%")

# Imprime la precisión anterior para comparación
print(f"Precisión conjunto prueba (modelo no balanceado): {test_acc:.2f}%")

Se comparan las precisiones del modelo original y del modelo balanceado, y se muestra la precisión por clase del modelo resampleado para evaluar su desempeño.

In [ ]:
print("Original test accuracy: {:.2f}%".format(test_acc))
print("Resampled test accuracy: {:.2f}%".format(test_acc_resampled))

# Analyze precision per class for the resampled model
print("\nPrecision per class (Resampled Model):")
for class_num, precision in accuracy_per_class_resampled.items():
    print(f"Class {class_num}: {precision:.4f}")

Se importa RandomForestClassifier de sklearn.ensemble para probar un modelo más complejo que pueda manejar mejor el desbalance de clases que la regresión logística.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

## Entrenar el modelo random forest
Se entrena un Random Forest con los datos originales, aprovechando que es menos sensible al desbalance que la regresión logística.


In [ ]:
# 1. Inicializa un objeto RandomForestClassifier. Puedes empezar con los parámetros por defecto.
# 2. Entrena el modelo utilizando el método .fit() con X_train y y_train.
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

**Evaluar el modelo random forest**


Se evalúa el Random Forest en el conjunto de prueba calculando la precisión general y mostrando un reporte de clasificación para observar el desempeño por clase.



In [ ]:
# 1. Utiliza el modelo rf_model entrenado para hacer predicciones en el conjunto de prueba X_test
pred_test_rf = rf_model.predict(X_test)

# 2. Calcula la precisión general del modelo Random Forest en el conjunto de prueba
test_acc_rf = np.mean(pred_test_rf == y_test) * 100

# Imprime el resultado formateado a dos decimales
print(f"Precisión conjunto prueba (Random Forest): {test_acc_rf:.2f}%")

# 3. Genera un reporte de clasificación utilizando la función classification_report
report_rf = classification_report(y_test, pred_test_rf, zero_division=0)

# 4. Imprime el reporte de clasificación generado
print("\nReporte de Clasificación (Random Forest):")
print(report_rf)

**Comparar resultados**


Se muestran las precisiones generales y los reportes de clasificación de los tres modelos para comparar su desempeño. Esto permite observar que Random Forest maneja mejor la complejidad y el desbalance de clases que la regresión logística, y que SMOTE no mejoró el modelo lineal.



In [ ]:
# 1. Imprime la precisión general del modelo de regresión logística original (`test_acc`).
print(f"Precisión conjunto prueba (Regresión Logística Original): {test_acc:.2f}%")

# 2. Imprime la precisión general del modelo de regresión logística con SMOTE (`test_acc_resampled`).
print(f"Precisión conjunto prueba (Regresión Logística con SMOTE): {test_acc_resampled:.2f}%")

# 3. Imprime la precisión general del modelo Random Forest (`test_acc_rf`).
print(f"Precisión conjunto prueba (Random Forest): {test_acc_rf:.2f}%")

# 4. Imprime el reporte de clasificación del modelo de regresión logística original
# The report for the original logistic regression model is available in the 'report' variable.
#print("\nReporte de Clasificación (Regresión Logística Original):")
#print(report)

# 5. Imprime el reporte de clasificación del modelo de regresión logística con SMOTE (`report_resampled`).
print("\nReporte de Clasificación (Regresión Logística con SMOTE):")
print(report_resampled)

# 6. Imprime el reporte de clasificación del modelo Random Forest (`report_rf`).
# The report for the Random Forest model is available in the 'report_rf' variable from the previous step.
print("\nReporte de Clasificación (Random Forest):")
print(report_rf)

# 7. Analiza y describe las diferencias clave en la precisión general y en el rendimiento por clase
print("\nAnálisis de la Comparación de Modelos:")
print("La precisión general en el conjunto de prueba para los tres modelos es la siguiente:")
print(f"- Regresión Logística Original: {test_acc:.2f}%")
print(f"- Regresión Logística con SMOTE: {test_acc_resampled:.2f}%")
print(f"- Random Forest: {test_acc_rf:.2f}%")

Esta versión de OneVsAll_with_history es igual que la función OneVsAll anterior, pero ahora devuelve también el historial de costos (J_history) de cada clase. Esto permite:

Visualizar cómo disminuye el costo durante el entrenamiento para cada clase.

Diagnosticar problemas de convergencia.

Comparar la rapidez de aprendizaje entre diferentes clases.

El retorno ahora incluye all_theta (los parámetros entrenados) y all_J_history (lista con los costos por iteración para cada clase).

In [ ]:
# Nueva version de la funcion OneVsAll que devuelve el historial de costos
def OneVsAll_with_history(X, y, num_labels, lambda_):
    alpha = 0.01
    num_iters = 20000

    m, n = X.shape
    all_theta = np.zeros((num_labels, n + 1))
    all_J_history = [] # Lista para almacenar el historial de costos de cada clase

    # Agrega unos a la matriz X
    Xb = np.concatenate([np.ones((m, 1)), X], axis=1)

    for c in np.arange(num_labels):
        initial_theta = np.zeros(n + 1)
        y_actual = np.where(y == c, 1, 0)
        theta, J_history = descensoGradiente(initial_theta, Xb, y_actual, alpha, num_iters) # Pasa Xb a descensoGradiente
        all_theta[c] = theta
        all_J_history.append(J_history) # Almacena el historial de costos para esta clase

    return all_theta, all_J_history  # Devuelve all_J_history

In [ ]:
num_labels = len(np.unique(y))
all_theta, all_J_history = OneVsAll_with_history(X, y, num_labels)

In [ ]:
# Justo antes de la celda que falla, añade:
print("Kernel activo. X_norm definido?", 'X_norm' in locals())


In [ ]:
def predictOneVsAll(all_theta, X):
    m = X.shape[0]
    Xb = np.concatenate([np.ones((m, 1)), X], axis=1)
    p = np.argmax(sigmoid(Xb.dot(all_theta.T)), axis=1)
    return p

y_pred = predictOneVsAll(all_theta, X)
train_acc = np.mean(y_pred == y) * 100
print(f"Precisión conjunto entrenamiento: {train_acc:.2f}%")  # Debe ser ~95.1%

Entrena One-vs-All con descenso de gradiente, guarda all_theta_gd y el historial de costos all_J_history, y grafica cómo converge el costo por clase.

In [ ]:
# Define el parámetro lambda
lambda_ = 0.001

# Usa la nueva función OneVsAll_with_history que devuelve el historial de costos
all_theta_gd, all_J_history = OneVsAll_with_history(X_train, y_train, num_labels, lambda_)

# Grafica el historial de costos para cada clase
pyplot.figure(figsize=(12, 8))
for c in range(num_labels):
    pyplot.plot(np.arange(len(all_J_history[c])), all_J_history[c], lw=1, label=f'Clase {c}')

pyplot.xlabel('Número de iteraciones')
pyplot.ylabel('Costo J')
pyplot.title('Convergencia del Costo por Clase (Descenso de Gradiente)')
# Reducir el número de etiquetas en la leyenda si hay muchas clases para evitar aglomeración
if num_labels <= 20:
    pyplot.legend()
else:
    # Puedes añadir una leyenda solo para algunas clases representativas si lo deseas
    pass # No mostrar leyenda para muchas clases por defecto

pyplot.grid(True)
pyplot.show()

print("All Theta (entrenado con Descenso de Gradiente):")
print(all_theta_gd)